In [1]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

/sw/spack-levante/mambaforge-23.1.0-1-Linux-x86_64-3boc6i/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


# Hybrid ML

In [3]:
#compute climatology from assimilation
path = '/work/uo1075/u241308/ML_infilling/tas/'
file ='asseikeraf_tas_r1-16i8p4_360x180.nc'

ref_min = 1985
ref_max = 2014

with xr.open_dataset(path+file,decode_times=False) as tas_clim:
    tas_clim = tas_clim.tas.mean(dim='sfc') #average over ensemble members
    #Time index is not readable for .dt. We create a new index
    reference_date = str('1958-1-01')
    print("Reference starting date: " + reference_date)
    tas_clim['time'] = pd.date_range(start=reference_date, periods=tas_clim.sizes['time'], freq='MS')
    
    tas_clim = tas_clim[(tas_clim.time.dt.year>=ref_min)&(tas_clim.time.dt.year<=ref_max)]
    tas_clim = tas_clim.groupby('time.month').mean(dim='time')

Reference starting date: 1958-1-01


In [4]:
path_in = '/work/uo1075/u241308/mpiesm-1.2.01p7-levante/proccessed_output/tas/'
path_out = '/work/uo1075/u241308/data_python_PostDoc/ML_assimilation/tas_ML_NA/anomaly/'
y_start = 1958
y_end = 2021

for y_id in range(y_start,y_end):
    file_in = 'tas_%i_r1-16i2p3-LR_26_months_360x180.nc' %y_id
    file_out = 'tas_%i_r1-16i2p3-LR_26_months_360x180_anomaly_ref_%i-%i.nc' %(y_id,ref_min,ref_max)
    with xr.open_dataset(path_in+file_in) as tas:
        tas = tas.tas
    tas_anomaly = tas.groupby('time.month') - tas_clim
    tas_anomaly.to_netcdf(path_out+file_out)
    print('Year %i done'%y_id)

Year 1958 done
Year 1959 done
Year 1960 done
Year 1961 done
Year 1962 done
Year 1963 done
Year 1964 done
Year 1965 done
Year 1966 done
Year 1967 done
Year 1968 done
Year 1969 done
Year 1970 done
Year 1971 done
Year 1972 done
Year 1973 done
Year 1974 done
Year 1975 done
Year 1976 done
Year 1977 done
Year 1978 done
Year 1979 done
Year 1980 done
Year 1981 done
Year 1982 done
Year 1983 done
Year 1984 done
Year 1985 done
Year 1986 done
Year 1987 done
Year 1988 done
Year 1989 done
Year 1990 done
Year 1991 done
Year 1992 done
Year 1993 done
Year 1994 done
Year 1995 done
Year 1996 done
Year 1997 done
Year 1998 done
Year 1999 done
Year 2000 done
Year 2001 done
Year 2002 done
Year 2003 done
Year 2004 done
Year 2005 done
Year 2006 done
Year 2007 done
Year 2008 done
Year 2009 done
Year 2010 done
Year 2011 done
Year 2012 done
Year 2013 done
Year 2014 done
Year 2015 done
Year 2016 done
Year 2017 done
Year 2018 done
Year 2019 done
Year 2020 done


# Standard

In [4]:
path_in = '/work/uo1075/u241308/data_python_PostDoc/ML_assimilation/tas_benchmark/processed/'
path_out = '/work/uo1075/u241308/data_python_PostDoc/ML_assimilation/tas_benchmark/anomaly/'
y_start = 1960
y_end = 2020

for y_id in range(y_start,y_end):
    file_in = 'tas_%i_r1-16i2p2-LR_3_years_360x180.nc' %y_id
    file_out = 'tas_%i_r1-16i2p2-LR_3_years_360x180_anomaly_ref_%i-%i.nc' %(y_id,ref_min,ref_max)
    with xr.open_dataset(path_in+file_in,decode_times=False) as tas:
        tas = tas.tas
        #time axis is wrong. we set a new one
        units, reference_date = tas.time.attrs['units'].split('since')
        list1 = list(reference_date)
        list1[9:11]='01'
        reference_date = ''.join(list1)
        tas['time'] = pd.date_range(start=reference_date, periods=tas.sizes['time'], freq='MS')
    tas_anomaly = tas.groupby('time.month') - tas_clim
    tas_anomaly.to_netcdf(path_out+file_out)
    print('Year %i done'%y_id)

Year 1960 done
Year 1961 done
Year 1962 done
Year 1963 done
Year 1964 done
Year 1965 done
Year 1966 done
Year 1967 done
Year 1968 done
Year 1969 done
Year 1970 done
Year 1971 done
Year 1972 done
Year 1973 done
Year 1974 done
Year 1975 done
Year 1976 done
Year 1977 done
Year 1978 done
Year 1979 done
Year 1980 done
Year 1981 done
Year 1982 done
Year 1983 done
Year 1984 done
Year 1985 done
Year 1986 done
Year 1987 done
Year 1988 done
Year 1989 done
Year 1990 done
Year 1991 done
Year 1992 done
Year 1993 done
Year 1994 done
Year 1995 done
Year 1996 done
Year 1997 done
Year 1998 done
Year 1999 done
Year 2000 done
Year 2001 done
Year 2002 done
Year 2003 done
Year 2004 done
Year 2005 done
Year 2006 done
Year 2007 done
Year 2008 done
Year 2009 done
Year 2010 done
Year 2011 done
Year 2012 done
Year 2013 done
Year 2014 done
Year 2015 done
Year 2016 done
Year 2017 done
Year 2018 done
Year 2019 done


# ERA5

In [60]:
path = '/work/uo1075/u241308/data_python_PostDoc/ML_assimilation/era5/'
file = 'era5_tas_1940-2023_360x180.nc'
file_out = 'era5_tas_1940-2023_360x180_anomaly_ref_%i-%i.nc' %(ref_min,ref_max)

era = xr.open_dataset(path+file).t2m
era_clim = era[(era.time.dt.year>=ref_min)&(era.time.dt.year<=ref_max)].groupby('time.month').mean(dim='time')
era_anomaly = era.groupby('time.month') - era_clim
era_anomaly.to_netcdf(path+file_out)